# PBS PrEP → HIV notifications: exploratory walkthrough

Reproducible, top-to-bottom walkthrough of the causal analysis described in `reports/full_narrative.md`. Run all cells in order.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))

import matplotlib.pyplot as plt
from src.utils.data_io import load_pbs_dispensing, load_hiv_notifications, load_state_covariates, PBS_LISTING_DATE, EPIC_NSW_START_DATE
from src.utils.plotting import set_style, STATE_COLORS, NATIONAL_COLOR

set_style()

## 1. Load the synthetic data

In [ ]:
pbs = load_pbs_dispensing()
hiv = load_hiv_notifications()
covariates = load_state_covariates()

pbs.head()

## 2. Stage 1 — interrupted time series on PBS dispensing

Sanity check: does national dispensing show the expected level shift at the April 2018 PBS listing?

In [ ]:
national = pbs.groupby('month', as_index=False)['prep_dispensing_count'].sum()

fig, ax = plt.subplots()
ax.plot(national['month'], national['prep_dispensing_count'])
ax.axvline(PBS_LISTING_DATE, color='black', linestyle='--', label='PBS listing (Apr 2018)')
ax.axvline(EPIC_NSW_START_DATE, color='grey', linestyle=':', label='EPIC-NSW start (Mar 2016)')
ax.set_title('National PBS PrEP dispensing')
ax.set_ylabel('Monthly dispensing count')
ax.legend()
plt.show()

See `src/analysis/01_interrupted_time_series.py` for the segmented regression fit on this series.

## 3. Stage 2 — difference-in-differences on HIV notifications

MSM notifications (treated group) vs. Heterosexual/Other (control group).

In [ ]:
trend = hiv.groupby(['quarter_start', 'transmission_category'], as_index=False)['hiv_notifications'].sum()

fig, ax = plt.subplots()
for cat, group in trend.groupby('transmission_category'):
    ax.plot(group['quarter_start'], group['hiv_notifications'], label=cat)
ax.axvline(PBS_LISTING_DATE, color='black', linestyle='--', label='PBS listing (Apr 2018)')
ax.set_title('National HIV notifications by transmission category')
ax.set_ylabel('Quarterly notifications')
ax.legend()
plt.show()

See `src/analysis/02_diff_in_diff.py` for the DiD regression (TODO: full model).

## 4. Stage 3 — staggered adoption (NSW vs. rest of Australia)

NSW's EPIC-NSW trial means it was effectively treated ~2 years before the rest of the country.

In [ ]:
msm = hiv[hiv['transmission_category'] == 'MSM']
msm_by_state = msm.groupby(['quarter_start', 'state'], as_index=False)['hiv_notifications'].sum()

fig, ax = plt.subplots()
for state, group in msm_by_state.groupby('state'):
    ax.plot(group['quarter_start'], group['hiv_notifications'], label=state, color=STATE_COLORS.get(state))
ax.axvline(EPIC_NSW_START_DATE, color='grey', linestyle=':', label='EPIC-NSW start (Mar 2016)')
ax.axvline(PBS_LISTING_DATE, color='black', linestyle='--', label='PBS listing (Apr 2018)')
ax.set_title('MSM HIV notifications by state')
ax.set_ylabel('Quarterly notifications')
ax.legend()
plt.show()

See `src/analysis/03_staggered_adoption.py` for the staggered-adoption DiD estimator (TODO: full model).

## 5. Stage 4 (optional) — heterogeneity by state

See `src/analysis/04_heterogeneity_causal_forest.py` for the CausalForestDML heterogeneity analysis (TODO: full model).

In [ ]:
covariates